# 리포트 해시 진단 재검토 — 해시 크기 개입의 효과와 남은 경로 진단

> ### 한 일
> **새 진단의 개별 자세 원장을 다시 집계하고 결론·비교 지표·후속 실험의 범위를 검토했다.**

### 결과
1. 완료된 4 [^1]설정·42 [^2]자세의 168 [^3]개 기록을 대조했다.
2. 선택된 회복 자세에서 hash 크기만 늘려 고립 판정과 참조 환경 경로가 회복된 결과를 확인했다.
3. 버퍼 단독 변경의 추가 회복은 없지만 경로 수와 작은 복소장 차이는 남는다.
4. 잔존 자세의 경로 클래스가 갈리므로 단일한 다른 원인으로 묶기 전에 세부 진단이 필요하다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 재집계 | 완료 원장 개별 행의 고립 판정·후보 수·복소장·클래스 합 대조 |
| 범위 | 저장된 simulation 자료의 CPU 재계산과 설계 검토. 새 GPU·RF 결과는 별도다. |

### 재현

```bash
/workspace/.venvs/py312/bin/python benchmark/review_hash_diagnostic_0916.py
```

| | |
|---|---|
| 출력 | `outputs/hash_diagnostic_review_0916.json` |
| 소요 | CPU 단일 코어 경량 재계산 |

---

## 현황 스냅샷

관측 시각: 2026-09-16T06:39:52+00:00 [^4]

[09-16 15:39:41] 상태 G0:3/0(상한0·남85G) G1:0/0(상한0·남40G·⛔보류) G2:0/0(상한0·남0G·⛔보류) G3:3/3(상한3·남0G) G4:3/3(상한3·남0G) · 큐 8/8 · 워커 9 · RAM 10.6G · CPU 0.58 [^5]

[09-16 15:39:42] 상태 G0:3/0(상한0·남85G) G1:0/0(상한0·남40G·⛔보류) G2:0/0(상한0·남0G·⛔보류) G3:3/3(상한3·남0G) G4:3/3(상한3·남0G) · 큐 5/8 · 워커 9 · RAM 10.6G · CPU 0.58 [^6]

각 supervisor의 worker 표시는 같은 전역 집계다. 큐 위치는 발주 진행이며 완료 결과 수와 구분한다. 실행·중단·GPU 보류 설정은 그대로 두었다.

## 진단이 실제로 보여 준 것

대상은 matrice4e [^7]·지면 장면·el -60 [^8]°·거리 15 [^9] m·carrier 3.5e+09 [^10] Hz·광선 4000000000 [^11]·깊이 2 [^12]의 선택 자세다.

| 설정 | 후보 버퍼 | hash 크기 | 회복 대상 중 고립 자세 | 최대 저장 후보 |
|---|---|---|---|---|
| production | 2000000 [^13] | 2000000 [^14] | 6 [^15]/6 [^16] | 28273 [^17] |
| hash8 | 2000000 [^18] | 8000000 [^19] | 1 [^20]/6 [^21] | 28509 [^22] |
| hash32 | 2000000 [^23] | 32000000 [^24] | 0 [^25]/6 [^26] | 28557 [^27] |
| both8 | 8000000 [^28] | 8000000 [^29] | 1 [^30]/6 [^31] | 28508 [^32] |
완료된 설정에서는 후보 버퍼 비포화가 기록됐다. 메모리 부족으로 끝나지 않은 both32는 이 표의 관측 범위 밖이다.

## 결론 문구: hash 크기의 개입 효과와 충돌 기전을 구분한다

버퍼를 고정한 hash 확대가 선택 자세의 회복을 이끌었다. 같은 hash에서 버퍼 확대의 추가 회복 수는 원장에 따로 기록했다. 이는 이전의 결합 cap ladder보다 강한 대조다.

권장 문장: «검사한 지면 셀의 선택 자세에서 후보 버퍼 포화 없이 hash 크기 확대가 참조 환경 경로를 회복시켰다. 해시 충돌에 의한 억제는 유력한 설명이며, 탈락한 후보와 회복 경로의 직접 대응은 추가 계측 대상이다.»

소스의 제한 문구도 같은 범위를 명시한다. 충돌 확정이라는 제목과 직접 충돌을 세지 않았다는 본문은 강도를 맞춘다.

근거: [benchmark/dropout_hash_vs_buffer_0916.py](../benchmark/dropout_hash_vs_buffer_0916.py) 행 42 [^33] · [benchmark/dropout_hash_vs_buffer_0916.py](../benchmark/dropout_hash_vs_buffer_0916.py) 행 415 [^34] · [benchmark/dropout_hash_vs_buffer_0916.py](../benchmark/dropout_hash_vs_buffer_0916.py) 행 436 [^35]

## 확인된 집계 문제: 누적 회복과 새 회복이 섞인다

hash8 → both8의 paired_contrasts.n_no_longer_isolated는 5 [^36]지만 실제 이 전환의 추가 회복은 0 [^37]이다.

현재 계산은 도착 설정이 생산 기준보다 회복됐는지만 센다. 출발 설정에서 이미 회복된 자세도 포함한다. 결과적으로 버퍼 변경의 효과를 이 열에서 읽으면 잘못 귀속할 수 있다.

누적 회복은 별도 열로 두고, 추가 회복은 출발에서 고립 AND 도착에서 정상인 자세를 센다. 반대 전환인 추가 악화도 함께 기록한다. 이번 버퍼 단독 대조에서 추가 환경 경로 회복이 없다는 결론은 유지된다.

근거: [benchmark/dropout_hash_vs_buffer_0916.py](../benchmark/dropout_hash_vs_buffer_0916.py) 행 1031 [^38]

## «버퍼 변경은 아무것도 바꾸지 않았다»의 범위

hash8 → both8에서 최대 |ΔE|/|median(E)|는 2.146e-05 [^39]이고, 선언한 이동 문턱은 2e-05 [^40]이다.

자세 8015 [^41]의 반환 경로 수는 2811 [^42] → 2810 [^43]다.
자세 8016 [^44]의 반환 경로 수는 2794 [^45] → 2793 [^46]다.

따라서 같은 것은 선택 자세의 회복 판정과 참조 환경 경로 회복 결과다. 작은 출력 차이를 버퍼 효과 또는 반복 변동으로 가르려면 같은 설정의 추가 반복이 필요하다. 차이를 관찰한 뒤 문턱을 올려 동등성을 선언하는 방식은 피한다.

## «핵 자세는 다른 현상»도 아직 분류 가설이다

전체 잔존 집합은 15 [^47]자세이고 이번 진단은 그중 6 [^48]자세를 골랐다. 환경 전용 경로가 있는 것은 5 [^49], 없는 것은 1 [^50]자세다.

환경 경로가 있는 표본에서는 이웃 평균 대비 복소장 차이를 주로 drone+environment 클래스가 설명하며, 그 클래스가 지배적인 표본은 5 [^51]개다.

이는 클래스별 복소수 합산 관찰이다. 물리 차폐·다중경로 간섭·다른 경로의 hash 억제는 추가로 구분할 가설이다. 환경 경로 하나의 부재 여부만으로 원인이 전부 다르다고 묶기보다 잔존 자세를 분류할 근거로 쓴다.

근거: [benchmark/dropout_hash_vs_buffer_0916.py](../benchmark/dropout_hash_vs_buffer_0916.py) 행 932 [^52]

## 표본·재현 범위는 이미 얻은 결과와 분리한다

생산 기준 재실행의 최대 정규화 오차는 1.973e-05 [^53], 고립 라벨 불일치는 0 [^54]다. 현재 자료는 선언한 오차 문턱 안에 들어간다.

회복 표본은 큰 cap에서 회복된 집합에서 미리 골랐다. 이 선택은 동일 자세의 개입 비교에 적합하지만 회복 확률의 무작위 표본이나 모든 장면의 성공률로 읽기 어렵다.

참조 경로 평가의 14 [^55]개 항목은 고유 자세 7 [^56]개의 이웃별 비교다. 이를 독립 자세 수로 세면 분모가 커진다.

후속 자동 실행에는 라벨 일치·복소장 허용오차·설정 read-back을 하드 관문으로 두고 실패 수를 선정 시점의 분모와 함께 남긴다. 제외 후 분모만 제시하면 재현 실패가 숨는다.

근거: [benchmark/dropout_hash_vs_buffer_0916.py](../benchmark/dropout_hash_vs_buffer_0916.py) 행 965 [^57]

## 안테나 큐의 판정도 출력 전체를 봐야 한다

현행 큐는 조준 안테나에서 고립 자세가 계속 없으면 안테나 축이 solver 설정과 독립이라고 읽는다. 고립 개수는 문턱 판정이므로 같은 개수가 같은 복소장·스펙트럼·검출 성능을 보장하지 않는다.

공통 자세 전체의 raw spectrum·rotor 대비·경로 클래스와 iso 대비 차이를 cap별로 평가한다. 정상 자세만 남긴 안정성과 전체 구간의 안정성을 따로 표시한다. 배경의 큰 정적 성분으로 나눈 작은 오차가 rotor 성분에는 클 수 있다.

안테나 이득의 변화량은 cap별 aimed−iso 지표 차이를 다시 비교하고, 사전 허용오차와 반복 변동을 함께 둔다. 판정이 유지돼도 범위는 검사한 장면·패턴·설정에 붙인다.

근거: [runners/jobs_0946_cap_antenna.txt](../runners/jobs_0946_cap_antenna.txt) 행 19 [^58] · [benchmark/dropout_knobs_0916.py](../benchmark/dropout_knobs_0916.py) 행 207 [^59]

## 다음 작업 순서

| 작업 | 우선순위와 종료 조건 |
|---|---|
| 원장·계획의 문구 및 paired 비교 집계 | 먼저 수행. 완료 설정·표본 분모·누적/추가 회복을 구분하면 독자가 같은 결과를 읽는다. |
| 잔존 자세의 세부 경로 | 전체 잔존 집합과 이웃·정상 대조에 한정. 클래스 차이와 경로 대응이 설명되는 지점에서 범위를 넓힐지 결정한다. |
| OFDM 수신기 pilot | CPU의 독립 합성 채널부터 병행. 정확한 delay/Doppler 복원과 H0 보정이 먼저다. |
| 더 큰 cap·새로운 대규모 sweep | 현재 안테나 비교와 경로 진단 결과를 읽은 뒤 필요성을 결정한다. |

## 잔존 자세 진단의 최소 산출물

전체 드론을 유지하고 잔존 자세·앞뒤 이웃·정상 대조에서 경로 ID와 중복 수, 지연, 복소 계수, interaction, primitive, 반사점 좌표를 저장한다. 같은 설정 반복으로 작은 차이의 범위를 먼저 얻는다.

환경 전용·드론 전용·혼합 경로를 나눠 복소장 차이의 잔차를 계산한 뒤, 큰 차이에 기여한 경로의 차폐와 경계 통과를 검사한다. 복소장 합산이 닫히는 것은 원인 입증과 구분한다.

현행 기본 실행은 선택 경로 설명과 클래스 집계를 보관했다. keep_paths 경로에도 전체 vertices 배열의 저장 여부를 확인해 필요한 필드를 보강해야 한다. GPU 진단은 공유 카드의 메모리 여유를 별도로 확보한 소규모 작업으로 편성한다.

근거: [benchmark/dropout_hash_vs_buffer_0916.py](../benchmark/dropout_hash_vs_buffer_0916.py) 행 899 [^60]

## OFDM pilot의 채널 가정을 명시한다

첫 단계는 알려진 합성 delay/Doppler 채널과 변하는 payload로 수신기를 점검한다. 사용 RE와 null RE를 구분하고 QPSK·QAM의 에너지 정규화, division·regularization의 잡음 효과를 비교한다.

H0 보정 데이터와 평가 H0/H1을 분리한다. 같은 관측시간·송신 에너지·잡음 조건에서 실제 Pfa·Pd·delay/Doppler 오차를 보고하며 truth는 신호 주입과 사후 점수에만 쓴다.

현재 합산 E(t)를 OFDM에 곱하면 flat-fading 가정을 둔 pilot이 된다. 거리 분해·주파수 선택성·파형 간 비교로 확장하려면 경로별 지연과 복소 계수, 시간 변화가 포함된 채널이 필요하다. 이 pilot을 완전한 Wi-Fi/LTE/NR 또는 실제 RF 검증으로 소개하지 않는다.

근거: [benchmark/design_isac_waveform_benchmark_0915.py](../benchmark/design_isac_waveform_benchmark_0915.py) 행 73 [^61]

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 누적/추가 회복과 결론 범위를 원 생성기에 반영한다 | 버퍼·hash의 효과를 같은 분모로 해석한다 | 진단 builder와 원장 |
| 잔존 집합의 경로별 자료와 같은 설정 반복을 확보한다 | 혼합 경로 변화와 반복 변동·가시성을 가른다 | 소규모 GPU 진단 |
| 합성 OFDM 수신기 pilot을 병행한다 | 신호 처리 오류를 장면 모델 오류와 분리한다 | CPU 수신기 시험 |
| 안테나 큐 완료 뒤 raw 지표의 교차 효과를 읽는다 | 패턴 효과의 적용 범위를 정한다 | 기존 실행 큐 결과 |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 61개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/hash_diagnostic_review_0916.json` | `grid_count` | 4 |
| [^2] | `outputs/hash_diagnostic_review_0916.json` | `pose_count` | 42 |
| [^3] | `outputs/hash_diagnostic_review_0916.json` | `row_count` | 168 |
| [^4] | `outputs/hash_diagnostic_review_0916.json` | `_meta.utc` | 2026-09-16T06:39:52+00:00 |
| [^5] | `outputs/hash_diagnostic_review_0916.json` | `queue.logs.sup_cap_0945_log` | [09-16 15:39:41] 상태 G0:3/0(상한0·남85G) G1:0/0(상한0·남40G·⛔보… |
| [^6] | `outputs/hash_diagnostic_review_0916.json` | `queue.logs.sup_cap_antenna_0946_log` | [09-16 15:39:42] 상태 G0:3/0(상한0·남85G) G1:0/0(상한0·남40G·⛔보… |
| [^7] | `outputs/hash_diagnostic_review_0916.json` | `cell.drone` | matrice4e |
| [^8] | `outputs/hash_diagnostic_review_0916.json` | `cell.el_deg` | -60 |
| [^9] | `outputs/hash_diagnostic_review_0916.json` | `cell.range_m` | 15 |
| [^10] | `outputs/hash_diagnostic_review_0916.json` | `cell.carrier_hz` | 3.5e+09 |
| [^11] | `outputs/hash_diagnostic_review_0916.json` | `cell.spp_used` | 4000000000 |
| [^12] | `outputs/hash_diagnostic_review_0916.json` | `cell.max_depth` | 2 |
| [^13] | `outputs/hash_diagnostic_review_0916.json` | `grids[0].buffer` | 2000000 |
| [^14] | `outputs/hash_diagnostic_review_0916.json` | `grids[0].hash` | 2000000 |
| [^15] | `outputs/hash_diagnostic_review_0916.json` | `grids[0].recovered_isolated` | 6 |
| [^16] | `outputs/hash_diagnostic_review_0916.json` | `grids[0].recovered_n` | 6 |
| [^17] | `outputs/hash_diagnostic_review_0916.json` | `grids[0].max_candidates` | 28273 |
| [^18] | `outputs/hash_diagnostic_review_0916.json` | `grids[1].buffer` | 2000000 |
| [^19] | `outputs/hash_diagnostic_review_0916.json` | `grids[1].hash` | 8000000 |
| [^20] | `outputs/hash_diagnostic_review_0916.json` | `grids[1].recovered_isolated` | 1 |
| [^21] | `outputs/hash_diagnostic_review_0916.json` | `grids[1].recovered_n` | 6 |
| [^22] | `outputs/hash_diagnostic_review_0916.json` | `grids[1].max_candidates` | 28509 |
| [^23] | `outputs/hash_diagnostic_review_0916.json` | `grids[2].buffer` | 2000000 |
| [^24] | `outputs/hash_diagnostic_review_0916.json` | `grids[2].hash` | 32000000 |
| [^25] | `outputs/hash_diagnostic_review_0916.json` | `grids[2].recovered_isolated` | 0 |
| [^26] | `outputs/hash_diagnostic_review_0916.json` | `grids[2].recovered_n` | 6 |
| [^27] | `outputs/hash_diagnostic_review_0916.json` | `grids[2].max_candidates` | 28557 |
| [^28] | `outputs/hash_diagnostic_review_0916.json` | `grids[3].buffer` | 8000000 |
| [^29] | `outputs/hash_diagnostic_review_0916.json` | `grids[3].hash` | 8000000 |
| [^30] | `outputs/hash_diagnostic_review_0916.json` | `grids[3].recovered_isolated` | 1 |
| [^31] | `outputs/hash_diagnostic_review_0916.json` | `grids[3].recovered_n` | 6 |
| [^32] | `outputs/hash_diagnostic_review_0916.json` | `grids[3].max_candidates` | 28508 |
| [^33] | `outputs/hash_diagnostic_review_0916.json` | `sources.scope.line` | 42 |
| [^34] | `outputs/hash_diagnostic_review_0916.json` | `sources.allocation.line` | 415 |
| [^35] | `outputs/hash_diagnostic_review_0916.json` | `sources.buffer.line` | 436 |
| [^36] | `outputs/hash_diagnostic_review_0916.json` | `buffer_comparison.ledger_n_no_longer_isolated` | 5 |
| [^37] | `outputs/hash_diagnostic_review_0916.json` | `buffer_comparison.incremental_recoveries` | 0 |
| [^38] | `outputs/hash_diagnostic_review_0916.json` | `sources.contrast.line` | 1031 |
| [^39] | `outputs/hash_diagnostic_review_0916.json` | `buffer_comparison.max_field_change` | 2.146e-05 |
| [^40] | `outputs/hash_diagnostic_review_0916.json` | `noise_tol` | 2e-05 |
| [^41] | `outputs/hash_diagnostic_review_0916.json` | `buffer_comparison.changed_path_counts[0].pose` | 8015 |
| [^42] | `outputs/hash_diagnostic_review_0916.json` | `buffer_comparison.changed_path_counts[0].npaths_from` | 2811 |
| [^43] | `outputs/hash_diagnostic_review_0916.json` | `buffer_comparison.changed_path_counts[0].npaths_to` | 2810 |
| [^44] | `outputs/hash_diagnostic_review_0916.json` | `buffer_comparison.changed_path_counts[1].pose` | 8016 |
| [^45] | `outputs/hash_diagnostic_review_0916.json` | `buffer_comparison.changed_path_counts[1].npaths_from` | 2794 |
| [^46] | `outputs/hash_diagnostic_review_0916.json` | `buffer_comparison.changed_path_counts[1].npaths_to` | 2793 |
| [^47] | `outputs/hash_diagnostic_review_0916.json` | `core_summary.population` | 15 |
| [^48] | `outputs/hash_diagnostic_review_0916.json` | `core_summary.sampled` | 6 |
| [^49] | `outputs/hash_diagnostic_review_0916.json` | `core_summary.with_env` | 5 |
| [^50] | `outputs/hash_diagnostic_review_0916.json` | `core_summary.without_env` | 1 |
| [^51] | `outputs/hash_diagnostic_review_0916.json` | `core_summary.both_dominant` | 5 |
| [^52] | `outputs/hash_diagnostic_review_0916.json` | `sources.path_ref.line` | 932 |
| [^53] | `outputs/hash_diagnostic_review_0916.json` | `replay.max_normalized_error` | 1.973e-05 |
| [^54] | `outputs/hash_diagnostic_review_0916.json` | `replay.label_mismatches` | 0 |
| [^55] | `outputs/hash_diagnostic_review_0916.json` | `reference_counts.neighbour_comparisons` | 14 |
| [^56] | `outputs/hash_diagnostic_review_0916.json` | `reference_counts.unique_poses` | 7 |
| [^57] | `outputs/hash_diagnostic_review_0916.json` | `sources.replay.line` | 965 |
| [^58] | `outputs/hash_diagnostic_review_0916.json` | `sources.queue.line` | 19 |
| [^59] | `outputs/hash_diagnostic_review_0916.json` | `sources.stability.line` | 207 |
| [^60] | `outputs/hash_diagnostic_review_0916.json` | `sources.save.line` | 899 |
| [^61] | `outputs/hash_diagnostic_review_0916.json` | `sources.pilot.line` | 73 |